# 3 · Integración de RAG

**Objetivo:** dar al agente acceso a conocimiento que el modelo no tiene, con una **recuperación
híbrida** (léxica + semántica) y, sobre todo, **integrada en el flujo del grafo**: recuperar, evaluar si
basta, reformular si no y responder citando fuentes.

RAG significa *Retrieval-Augmented Generation*: generación aumentada con recuperación de información.

**Empezar aquí sin haber hecho las secciones anteriores:** solo necesitas tu `.env`. Todo lo demás se
importa del paquete `taller`.

In [ ]:
from langchain_core.documents import Document

from taller import get_embeddings, get_llm
from taller.kb import format_docs, load_knowledge_base
from taller.viz import mostrar_grafo

llm = get_llm()

## 3.1 · El problema

Cuac & Co. es una empresa ficticia. El modelo no puede saber nada de ella.

In [ ]:
llm.invoke("En Cuac & Co., ¿qué significa el error ERR-4012 y qué hay que hacer?").text

Puede que invente algo verosímil (una **alucinación**) o que admita que no lo sabe. En ambos casos,
necesitamos darle el contexto.

## 3.2 · La base de conocimiento

10 documentos Markdown en `datos/conocimiento/`. Los dividimos en **un fragmento por sección `##`** y
añadimos el título del documento a cada fragmento para que se entienda por sí solo.

In [ ]:
docs = load_knowledge_base()
print(f"{len(docs)} fragmentos\n")
print(docs[5].page_content)

## 3.3 · Dos formas de buscar

| | **BM25 (léxica)** | **Embeddings (semántica)** |
|---|---|---|
| Compara | palabras exactas, ponderadas por rareza | significado (vectores) |
| Brilla con | códigos, nombres propios: `ERR-4012`, `Chapuzón` | paráfrasis: «vacaciones» ≈ «días de descanso» |
| Falla con | sinónimos | identificadores raros |

En `taller/retrieval.py` hay un BM25 de unas 30 líneas, escrito a mano.

In [ ]:
from taller.retrieval import BM25Index, DenseIndex

bm25 = BM25Index(docs)
embeddings = get_embeddings()
dense = DenseIndex(docs, embeddings) if embeddings else None


def comparar(query: str, k: int = 3) -> None:
    print(f"Consulta: {query!r}")
    print("  BM25      →", [d.id for d, _ in bm25.search(query, k)])
    if dense:
        print("  Semántica →", [d.id for d, _ in dense.search(query, k)])


comparar("ERR-4012")
comparar("me quiero ir de vacaciones dos semanas, ¿cómo lo pido?")

Fíjate en que la documentación nunca usa la palabra «vacaciones» (habla de «días de descanso» y
«ausencias»). Y que un código como `ERR-4012` no tiene «significado» para un modelo de embeddings.

## 3.4 · Búsqueda híbrida con *Reciprocal Rank Fusion* (RRF)

¿Cómo combinamos dos rankings si sus puntuaciones no son comparables? **Usando solo las posiciones**:

$$\text{score}(d) = \sum_{\text{rankings}} \frac{1}{k + \text{posición}(d)} \qquad (k = 60)$$

Un documento que aparece arriba en las dos listas gana; uno que solo aparece en una, también puntúa.

### ✍️ Ejercicio 3: implementa RRF

In [ ]:
def reciprocal_rank_fusion(rankings: list[list[Document]], k: int = 60) -> list[Document]:
    scores: dict[str, float] = {}
    by_id: dict[str, Document] = {}
    for ranking in rankings:
        for position, doc in enumerate(ranking, start=1):
            by_id[doc.id] = doc
            # TODO: suma 1 / (k + position) a scores[doc.id]
            ...
    # TODO: devuelve los documentos ordenados por puntuación descendente
    ...

<details><summary><b>Ver solución</b></summary>

```python
def reciprocal_rank_fusion(rankings: list[list[Document]], k: int = 60) -> list[Document]:
    scores: dict[str, float] = {}
    by_id: dict[str, Document] = {}
    for ranking in rankings:
        for position, doc in enumerate(ranking, start=1):
            by_id[doc.id] = doc
            scores[doc.id] = scores.get(doc.id, 0.0) + 1 / (k + position)
    return [by_id[i] for i in sorted(scores, key=scores.get, reverse=True)]
```
</details>

Si te atascas, la versión de referencia está en `taller.retrieval.reciprocal_rank_fusion` (devuelve pares
`(documento, puntuación)`; aquí solo documentos) y la usa `HybridRetriever`, que es lo que empleamos a partir de aquí. Sin embeddings (por ejemplo, con Anthropic) degrada a solo BM25.

In [ ]:
# Comprueba tu RRF: ERR-4012 debe seguir arriba aunque la semántica no lo entienda.
rankings = [[d for d, _ in bm25.search("ERR-4012", 3)]]
if dense:
    rankings.append([d for d, _ in dense.search("ERR-4012", 3)])
try:
    print([d.id for d in reciprocal_rank_fusion(rankings)][:3])
except TypeError:
    print("Completa el ejercicio de arriba (o copia la solución) y vuelve a ejecutar esta celda.")

In [ ]:
from taller.retrieval import HybridRetriever

retriever = HybridRetriever(docs, embeddings)
for d in retriever.search("¿Puedo desplegar un viernes por la tarde?", k=3):
    print("·", d.id)

## 3.5 · RAG como parte del grafo, no como una llamada aislada

El RAG «de ejemplo» es `contexto = buscar(pregunta); responder(contexto)`. En sistemas reales,
la primera búsqueda a menudo no basta. Lo modelamos como un grafo con un **bucle correctivo**:

1. `retrieve` recupera fragmentos.
2. `grade` pide al LLM que evalúe si bastan (con **salida estructurada**: un objeto Pydantic, no texto libre).
3. Si no bastan, se reformula la consulta y se vuelve a recuperar (como mucho 2 intentos).
4. `generate` responde **solo** con el contexto y cita fuentes.

In [ ]:
from typing import Literal, TypedDict

from langchain_core.messages import SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


class RAGState(TypedDict):
    question: str
    query: str
    documents: list[Document]
    answer: str
    attempts: int


class Grade(BaseModel):
    """Evaluación de si los documentos bastan para responder."""

    suficiente: bool = Field(description="True si los documentos permiten responder a la pregunta")
    nueva_consulta: str = Field(
        description="Si no bastan, una consulta de búsqueda alternativa; si bastan, vacío"
    )


grader = llm.with_structured_output(Grade)


def retrieve(state: RAGState) -> dict:
    query = state.get("query") or state["question"]
    return {"documents": retriever.search(query, k=4), "attempts": state.get("attempts", 0) + 1}


def grade(state: RAGState) -> dict:
    verdict = grader.invoke(
        f"Pregunta: {state['question']}\n\nDocumentos:\n{format_docs(state['documents'])}\n\n"
        "¿Contienen la información necesaria para responder? Si no, propón otra consulta de búsqueda."
    )
    # verdict es None si el modelo no emite la llamada estructurada: seguimos con lo que hay
    return {"query": "" if verdict is None or verdict.suficiente else verdict.nueva_consulta}


def route(state: RAGState) -> Literal["retrieve", "generate"]:
    return "retrieve" if state["query"] and state["attempts"] < 2 else "generate"


def generate(state: RAGState) -> dict:
    system = (
        "Eres el asistente interno de Cuac & Co.. Responde en español usando SOLO el contexto. "
        "Cita la fuente entre corchetes. Si el contexto no contiene la respuesta, dilo.\n\n"
        f"Contexto:\n{format_docs(state['documents'])}"
    )
    return {"answer": llm.invoke([SystemMessage(system), ("human", state["question"])]).text}


builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("grade", grade)
builder.add_node("generate", generate)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", route)
builder.add_edge("generate", END)
rag = builder.compile()

mostrar_grafo(rag)

In [ ]:
resultado = rag.invoke({"question": "En Cuac & Co., ¿qué significa el error ERR-4012 y qué hay que hacer?"})
print(resultado["answer"])

Vamos a ver el **recorrido** por el grafo con `stream`: qué nodos se ejecutan y con qué consulta.

In [ ]:
def recorrido(pregunta: str) -> None:
    for part in rag.stream({"question": pregunta}, stream_mode="updates"):
        for nodo, update in part.items():
            detalle = ""
            if nodo == "retrieve":
                detalle = f"→ {[d.id for d in update['documents']]}"
            elif nodo == "grade":
                detalle = f"→ nueva consulta: {update['query']!r}" if update["query"] else "→ suficiente"
            elif nodo == "generate":
                detalle = f"\n{update['answer']}"
            print(f"[{nodo}] {detalle}")


recorrido("Me han despertado a las 3 de la mañana por una alerta. ¿A qué hora entro mañana?")

**En Studio:** el mismo grafo está servido como **rag** (`langgraph.json` → `taller/studio.py`). Con el
servidor arrancado (`uv run studio`), elígelo, pon una pregunta en `question` y mira el recorrido:
`retrieve` → `grade` → (`retrieve` otra vez si no basta) → `generate`. En cada nodo puedes abrir el estado y ver
los fragmentos recuperados y la nota de `grade`.

### ✍️ Ejercicio 4

Prueba `recorrido(...)` con preguntas que **no** estén en la documentación (por ejemplo, *«¿Cuál es el
menú de la cafetería?»*). ¿Pasa el grafo dos veces por `retrieve`? ¿Qué responde `generate`?

Después, prueba una pregunta que mezcle dos documentos: *«¿Cuánto me pagan de dietas si voy a dar un
taller a PyConES y cuenta para mi presupuesto de formación?»*

In [ ]:
# TODO: tus preguntas

## 3.6 · ¿RAG como nodo o como herramienta?

| | **RAG como nodo** (esta sección) | **RAG como herramienta** (sección 4) |
|---|---|---|
| ¿Quién decide buscar? | El grafo: **siempre** busca | El LLM: busca **si lo cree necesario** |
| Ventaja | Predecible, fácil de evaluar | Flexible: varias búsquedas, conversación |
| Riesgo | Busca cuando no hace falta | Que no busque y se lo invente |

Lo habitual en producción es una mezcla: herramientas para la conversación y nodos obligatorios donde
hay que garantizar algo (por ejemplo, recuperar siempre antes de responder sobre políticas).

> **Para recordar:** BM25 + embeddings + RRF es una base híbrida robusta y sin magia. Y el valor de
> LangGraph está en **orquestar** la recuperación: evaluar, reintentar, enrutar y citar.

### 🚀 Reto opcional

Añade un nodo `rewrite` **antes** de `retrieve` que convierta la pregunta del usuario en una consulta de
búsqueda (palabras clave, sin cortesías). ¿Mejoran los resultados de BM25?